# SAM-CREPE step 1: SAM-Whistle's confidence maps

Runs the published, already-trained SAM-Whistle model (Zhang et al. 2025) on the windows cut by `prepare_inputs.py cut`, and saves its **confidence map** for each window. Nothing is trained, and no graph search or scoring is done here: SAM-CREPE does those on your computer from the maps.

Only the model needs a GPU. First: *Runtime, Change runtime type, **T4 GPU***.

1. Run cells 1 and 2 (install, model weights).
2. Drag `SAM-CREPE/inputs_<name>/sam_whistle_upload.zip` into the **Files** panel (folder icon, left).
3. Set `UPLOAD` and `LABEL` in cell 3, then run cells 3 to 6.
4. Cell 6 downloads `conf_maps_<LABEL>.zip`. On your computer:
   `SAM-CREPE\.venv\Scripts\python SAM-CREPE\prepare_inputs.py collect --inputs SAM-CREPE\inputs_<name> --maps conf_maps_<LABEL>.zip`

Made from use case C of `notebooks/01_reproduce_sam_whistle_dclde.ipynb`, which made the maps of the 33 development selections, with SAM-Whistle pinned to commit 8b48829, the same code as `SAM-CREPE/external/SAM-Whistle/`. Colab deletes everything when the session ends, so download the maps before closing it.

## 1. Install

- **Java**: SAM-Whistle's graph search is silbido's Java code, called from Python through JPype (installed so that SAM-Whistle imports; the maps themselves do not use it).
- **SAM-Whistle** at commit 8b48829, and Meta's **segment-anything**. The SAM-Whistle repository contains an *empty* `segment-anything` folder (a git submodule placeholder), so it is replaced with a real clone.

Colab's `!` commands don't stop the notebook when they fail, so the last line checks that everything imports.

In [ ]:
import os
SAM_WHISTLE_COMMIT = '8b48829203a1427e985dddae2619d862ad1395cd'   # the same code as SAM-CREPE/external/SAM-Whistle
!apt-get -qq install -y openjdk-17-jre-headless > /dev/null
if not os.path.exists('/content/sam_whistle'):
    !git clone -q https://github.com/xixinzhang/sam_whistle.git /content/sam_whistle
%cd /content/sam_whistle
!git checkout -q {SAM_WHISTLE_COMMIT}
if not os.path.exists('segment-anything/setup.py'):
    !rm -rf segment-anything
    !git clone -q https://github.com/facebookresearch/segment-anything
!pip install -q -e . -e ./segment-anything
# new installs are visible to fresh Python processes (cell 5 uses one), so check there
check = !python -c "import segment_anything, sam_whistle, jpype, tyro; print('imports ok')"
print(check[-1])
assert check and check[-1] == 'imports ok', 'Install failed: scroll up for the pip error'

## 2. Model weights

- `sam_checkpoints/sam_vit_b_01ec64.pth`: Meta's base SAM ViT-B weights, which the model is built from.
- `logs/sam/img_encoder.pth` and `logs/sam/decoder.pth`: the authors' whistle-trained weights (their Google Drive folder, linked in the SAM-Whistle README). These replace the base weights.

In [ ]:
import glob, shutil
os.makedirs('sam_checkpoints', exist_ok=True)
if not os.path.exists('sam_checkpoints/sam_vit_b_01ec64.pth'):
    !wget -q -P sam_checkpoints https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
if not os.path.exists('logs/sam/img_encoder.pth'):
    !gdown -q --folder https://drive.google.com/drive/folders/1LXkczIaOyIMcu4Zmiamgc6jbtw6rrupC -O logs/sam
    # gdown sometimes puts files in a subfolder; the code expects them directly in logs/sam
    for f in glob.glob('logs/sam/**/*.pth', recursive=True) + glob.glob('logs/sam/**/configs.json', recursive=True):
        if os.path.dirname(f) != 'logs/sam':
            shutil.move(f, 'logs/sam/')
print(sorted(os.listdir('logs/sam')))
assert os.path.exists('logs/sam/img_encoder.pth') and os.path.exists('logs/sam/decoder.pth')

## 3. Your upload

Drag `sam_whistle_upload.zip` (from `SAM-CREPE/inputs_<name>/`) into the **Files** panel first: it lands in `/content/`. Set `LABEL` to the inputs folder's name, so the download is called `conf_maps_<LABEL>.zip`.

In [ ]:
%cd /content/sam_whistle
UPLOAD = '/content/sam_whistle_upload.zip'   # where the dragged-in zip is
LABEL = 'unseen'                             # the <name> of SAM-CREPE/inputs_<name>
print('upload:', UPLOAD, '  label:', LABEL)

## 4. Unpack the upload

Unpacks the zip into `data/upload_<LABEL>/` and finds the folder holding `audio/` (the zip from `prepare_inputs.py` has `<name>/audio/<window>.wav` and `<name>/meta.json`). Warns about any window that is not 192 kHz or is shorter than 3 s: the model was trained on 192 kHz spectrograms, and SAM-Whistle fails on audio shorter than one 3 s block.

In [ ]:
import json, pathlib, zipfile
import soundfile as sf

assert os.path.exists(UPLOAD), (f'{UPLOAD} not found. Drag the zip into the Files panel and set UPLOAD in cell 3. '
                                f'Found in /content: {sorted(os.listdir("/content"))}')
dest = pathlib.Path(f'data/upload_{LABEL}')
if dest.exists():
    shutil.rmtree(dest)
zipfile.ZipFile(UPLOAD).extractall(dest)
roots = [p.parent for p in dest.rglob('audio') if p.is_dir()]
assert len(roots) == 1, f'Expected one audio/ folder in the upload, found: {roots}'
ROOT = str(roots[0])
if not os.path.exists(f'{ROOT}/meta.json'):
    stems = sorted(w.stem for w in pathlib.Path(ROOT, 'audio').glob('*.wav'))
    json.dump({'train': [], 'test': stems}, open(f'{ROOT}/meta.json', 'w'), indent=1)
meta = json.load(open(f'{ROOT}/meta.json'))
STEMS = meta['test'] + meta['train']
assert STEMS, f'No windows listed in {ROOT}/meta.json'
for stem in STEMS:
    info = sf.info(f'{ROOT}/audio/{stem}.wav')
    if info.samplerate != 192000 or info.duration < 3:
        print(f'WARNING {stem}: {info.samplerate} Hz, {info.duration:.3f} s (the model expects 192 kHz, at least 3 s)')
print(f'{len(STEMS)} windows in {ROOT}')

## 5. Make and save the maps

SAM-Whistle's own evaluation script computes the confidence map (`TonalTracker.sam_inference()`) but never saves it. This short script does the same inference and saves the map, with the same spectrogram settings as every map SAM-CREPE has used. It runs as a separate Python process, because the packages installed in cell 1 are only visible to new processes.

In [ ]:
%%writefile save_conf_maps.py
"""Save SAM-Whistle's confidence map for every recording in <root_dir>/meta.json (test + train).

Writes <out_dir>/<stem>_conf.npz with:
  conf     [freq rows x frames] float32, model confidence 0-1. Row 0 = lowest frequency, as graph search
           reads it (TonalTracker.build_graph / _select_peaks).
  freq_hz  frequency of each row: 5000 + 125 * row (the 5-50 kHz crop)
  time_s   time of each frame = frame START (--spect_cfg.no_center); the 8 ms of audio a frame covers is
           centred 4 ms later.
Usage: python save_conf_maps.py OUT_DIR <the same TonalConfig flags as eval_tonal.py>
"""
import json
import os
import sys

import numpy as np
import tyro

from sam_whistle.config import TonalConfig
from sam_whistle.evaluate.tonal_extraction.tonal_tracker import TonalTracker

out_dir = sys.argv[1]
cfg = tyro.cli(TonalConfig, args=sys.argv[2:])
os.makedirs(out_dir, exist_ok=True)
meta = json.load(open(os.path.join(cfg.root_dir, cfg.meta_file)))
for stem in meta['test'] + meta['train']:
    tracker = TonalTracker(cfg, stem)
    tracker.sam_inference()                      # replaces tracker.spect_map with the model's confidence
    conf = tracker.spect_map.astype(np.float32)
    np.savez_compressed(os.path.join(out_dir, f'{stem}_conf.npz'), conf=conf,
                        freq_hz=tracker.offset_Hz + np.arange(tracker.H) * tracker.freq_bin,
                        time_s=tracker.start_s + np.arange(tracker.W) * tracker.hop_s)
    print(f'saved {stem}: {conf.shape[0]} rows x {conf.shape[1]} frames, max confidence {conf.max():.2f}')

In [ ]:
OUT_MAPS = f'conf_maps/{LABEL}'
!python save_conf_maps.py {OUT_MAPS} --use_conf --root_dir {ROOT} \
    --spect_cfg.normalize zscore --spect_cfg.no_center --spect_cfg.interp linear --spect_cfg.kernel_size 3 \
    --log_dir logs/sam
saved = glob.glob(f'{OUT_MAPS}/*_conf.npz')
assert len(saved) == len(STEMS), f'Only {len(saved)} of {len(STEMS)} maps saved: scroll up for the Python error'
print(len(saved), 'maps saved in', OUT_MAPS)

## 6. Download the maps

Zips the maps into `/content/conf_maps_<LABEL>.zip` and starts a browser download. If it doesn't start, open the **Files** panel, find the zip under `/content/`, and choose *Download* from its menu, before the session ends.

Then, on your computer: `SAM-CREPE\.venv\Scripts\python SAM-CREPE\prepare_inputs.py collect --inputs SAM-CREPE\inputs_<name> --maps <path to conf_maps_<LABEL>.zip>`

In [ ]:
from google.colab import files
archive = shutil.make_archive(f'/content/conf_maps_{LABEL}', 'zip', OUT_MAPS)
print(f'{archive}: {len(glob.glob(f"{OUT_MAPS}/*_conf.npz"))} maps, {os.path.getsize(archive) / 1e6:.0f} MB')
files.download(archive)